# T5Gemma 2 1B–1B: small encoder-decoder drafter pilot

This Colab notebook implements the plan's **final-encoder-memory** pilot. It freezes `google/t5gemma-2-1b-1b`, trains a small causal drafter to imitate the frozen model's own greedy continuations, then checks that block verification reproduces ordinary greedy output.

- Dataset: a deterministic, streamed slice of CNN/DailyMail 3.0.0 (128 train articles, 16 validation articles). The dataset references are shown only for inspection; they are not used as distillation targets.
- Only the drafter projections, Transformer blocks, and positional embeddings are optimized. The target encoder, decoder, and shared token embedding/output head remain frozen.
- This is a correctness and feasibility pilot. It does **not** implement intermediate encoder taps, encoder/drafter overlap, request batching, or a KV-cache-aware verifier, and it makes no production speedup claim.
- The model is gated. Accept Google's Gemma terms for the checkpoint on Hugging Face before running the download cell.


In [ ]:
# Colab setup. This installs Python packages in the Colab runtime, not on your Mac.
%pip install -q "transformers>=4.57,<6" "datasets>=3,<6" "accelerate>=1" sentencepiece "huggingface_hub>=1.32,<2"

In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login, notebook_login, whoami
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers.modeling_outputs import BaseModelOutput

SEED = 42
MODEL_ID = "google/t5gemma-2-1b-1b"
DATASET_ID = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"
TRAIN_LIMIT = 128
VALIDATION_LIMIT = 16
SOURCE_MAX_TOKENS = 512
TEACHER_MAX_NEW_TOKENS = 64
DRAFT_MAX_NEW_TOKENS = 64
DRAFT_K = 4
DRAFT_WIDTH = 256
DRAFT_LAYERS = 2
DRAFT_HEADS = 4
DRAFT_FFN_WIDTH = 1024
KD_TOP_K = 16
KD_TEMPERATURE = 2.0
KD_WEIGHT = 0.25
TRAIN_EPOCHS = 1
LEARNING_RATE = 3e-4
PARITY_EVAL_EXAMPLES = 4
OUTPUT_DIR = Path("/content/t5gemma2_drafter")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime: Runtime > Change runtime type > GPU.")
DEVICE = torch.device("cuda")
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0))
print("Target dtype:", DTYPE)
print("Output directory:", OUTPUT_DIR)

## Authenticate to Hugging Face

If the model is not already accessible in this Colab session, enter a Hugging Face token with read access. The token is requested interactively and is not written into the notebook.

In [ ]:
try:
    whoami()
    print("Hugging Face session is already authenticated.")
except Exception:
    notebook_login()

## Load a small streamed dataset slice

CNN/DailyMail is a standard English news summarization dataset. Streaming avoids downloading entire splits just to select a few rows. We use article text as the source and let the frozen target generate the continuations used to train the drafter.

In [ ]:
train_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="train", streaming=True).shuffle(
    seed=SEED, buffer_size=2048
)
validation_stream = load_dataset(DATASET_ID, DATASET_CONFIG, split="validation", streaming=True)
train_rows = list(train_stream.take(TRAIN_LIMIT))
validation_rows = list(validation_stream.take(VALIDATION_LIMIT))
assert len(train_rows) == TRAIN_LIMIT, f"Expected {TRAIN_LIMIT} streamed train rows; got {len(train_rows)}"
assert len(validation_rows) == VALIDATION_LIMIT, (
    f"Expected {VALIDATION_LIMIT} validation rows; got {len(validation_rows)}"
)

PROMPT_PREFIX = "Summarize the following news article:\n"
train_sources = [PROMPT_PREFIX + row["article"].strip() for row in train_rows]
validation_sources = [PROMPT_PREFIX + row["article"].strip() for row in validation_rows]
print(f"Selected {len(train_sources)} train and {len(validation_sources)} validation articles.")
print("Dataset columns:", sorted(train_rows[0].keys()))
print("Source preview:\n", train_rows[0]["article"][:600])
print("Reference preview (inspection only, excluded from distillation):\n", train_rows[0]["highlights"][:300])

## Load the frozen T5Gemma2 target

The 1B–1B checkpoint is downloaded from Hugging Face after authentication. No quantization is applied, so the target logits used for distillation and verification come from the selected BF16/FP16 weights.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
target = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
).to(DEVICE)
target.eval()
for parameter in target.parameters():
    parameter.requires_grad_(False)

shared_embedding = target.get_input_embeddings()
shared_lm_head = target.get_output_embeddings()
if shared_embedding is None or shared_lm_head is None:
    raise RuntimeError("This checkpoint must expose input embeddings and an output head for the shared-weight drafter.")

encoder = target.get_encoder()
start_id = None
for candidate in (
    target.generation_config.decoder_start_token_id,
    getattr(target.config.decoder, "decoder_start_token_id", None),
    target.generation_config.bos_token_id,
    tokenizer.bos_token_id,
    tokenizer.pad_token_id,
):
    if candidate is not None:
        start_id = int(candidate)
        break
if start_id is None:
    raise RuntimeError("Could not determine the decoder start token ID.")
eos_ids = target.generation_config.eos_token_id
if eos_ids is None:
    eos_ids = target.config.eos_token_id
EOS_IDS = set(eos_ids if isinstance(eos_ids, (list, tuple)) else [eos_ids])
EOS_IDS.discard(None)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else start_id

print("Tokenizer vocabulary:", len(tokenizer))
print("Input embedding shape:", tuple(shared_embedding.weight.shape))
print("Output head:", shared_lm_head)
print("Decoder start ID:", start_id, "EOS IDs:", sorted(EOS_IDS))
print("Frozen target parameters:", f"{sum(p.numel() for p in target.parameters()):,}")
print("Target resident VRAM (GiB):", round(torch.cuda.memory_allocated() / 1024**3, 2))

## Tokenize sources and collect teacher continuations

We generate greedy continuations from the frozen target and store only their token IDs. This keeps the notebook's training slice small and avoids using human references as teacher labels. The generation pass is intentionally one source at a time to keep peak memory predictable on Colab GPUs.

In [ ]:
def tokenize_source(text):
    encoded = tokenizer(
        text,
        max_length=SOURCE_MAX_TOKENS,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


@torch.inference_mode()
def generate_teacher_trace(source_text):
    batch = tokenize_source(source_text)
    generated = target.generate(
        **batch,
        max_new_tokens=TEACHER_MAX_NEW_TOKENS,
        do_sample=False,
        num_beams=1,
        use_cache=True,
    )[0].tolist()
    if not generated or generated[0] != start_id:
        generated = [start_id] + generated
    for index, token_id in enumerate(generated[1:], start=1):
        if token_id in EOS_IDS:
            generated = generated[: index + 1]
            break
    return generated


train_traces = []
for source in tqdm(train_sources, desc="Teacher train traces"):
    train_traces.append(generate_teacher_trace(source))
validation_traces = []
for source in tqdm(validation_sources, desc="Teacher validation traces"):
    validation_traces.append(generate_teacher_trace(source))

trace_path = OUTPUT_DIR / "teacher_traces.jsonl"
with trace_path.open("w", encoding="utf-8") as handle:
    for split, sources, traces in (
        ("train", train_sources, train_traces),
        ("validation", validation_sources, validation_traces),
    ):
        for index, (source, trace) in enumerate(zip(sources, traces)):
            handle.write(
                json.dumps({"split": split, "index": index, "source": source, "teacher_ids": trace}, ensure_ascii=False)
                + "\n"
            )
print("Saved teacher traces:", trace_path)
print("Mean teacher continuation tokens:", round(sum(max(0, len(x) - 1) for x in train_traces) / len(train_traces), 1))

## Define the drafter

The draft model is a small causal Transformer with cross-attention to the target encoder's **final** memory. It uses the target's frozen token embedding and vocabulary head through trainable projections. Those large target weights are not registered inside the drafter, so drafter checkpoints contain only newly trained parameters.

In [ ]:
class SourceConditionedDrafter(nn.Module):
    def __init__(self, encoder_width, shared_width, width, layers, heads, ffn_width, max_positions):
        super().__init__()
        if width % heads:
            raise ValueError("DRAFT_WIDTH must be divisible by DRAFT_HEADS")
        self.width = width
        self.max_positions = max_positions
        self.source_projection = nn.Linear(encoder_width, width)
        self.token_projection = nn.Linear(shared_width, width)
        self.position_embedding = nn.Embedding(max_positions, width)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=width,
            nhead=heads,
            dim_feedforward=ffn_width,
            dropout=0.0,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )
        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=layers,
            norm=nn.LayerNorm(width),
        )
        self.output_projection = nn.Linear(width, shared_width)
        self.reset_parameters()

    def reset_parameters(self):
        for module in (self.source_projection, self.token_projection, self.output_projection):
            nn.init.xavier_uniform_(module.weight)
            nn.init.zeros_(module.bias)
        nn.init.normal_(self.position_embedding.weight, mean=0.0, std=0.02)

    def forward(self, decoder_ids, decoder_mask, source_memory, source_mask, input_embedding, output_head):
        batch_size, target_length = decoder_ids.shape
        if target_length > self.max_positions:
            raise ValueError(f"Decoder prefix {target_length} exceeds drafter position limit {self.max_positions}")
        token_vectors = input_embedding(decoder_ids).float()
        target_vectors = self.token_projection(token_vectors)
        positions = torch.arange(target_length, device=decoder_ids.device)
        target_vectors = target_vectors + self.position_embedding(positions)[None, :, :]
        memory = self.source_projection(source_memory.float())
        causal_mask = torch.triu(
            torch.ones(target_length, target_length, dtype=torch.bool, device=decoder_ids.device),
            diagonal=1,
        )
        hidden = self.decoder(
            tgt=target_vectors,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=~decoder_mask.bool(),
            memory_key_padding_mask=~source_mask.bool(),
        )
        shared_width = output_head.weight.shape[1]
        projected = self.output_projection(hidden).to(output_head.weight.dtype)
        if projected.shape[-1] != shared_width:
            raise RuntimeError("Drafter output projection does not match the target vocabulary head width")
        return output_head(projected)


encoder_config = getattr(target.config, "encoder", None)
encoder_text_config = getattr(encoder_config, "text_config", encoder_config)
encoder_width = getattr(encoder_text_config, "hidden_size", None)
if encoder_width is None:
    encoder_width = getattr(target.config, "d_model", None) or getattr(target.config, "hidden_size", None)
if encoder_width is None:
    raise RuntimeError("Could not determine the target encoder hidden width.")
shared_width = int(shared_embedding.weight.shape[1])
drafter = SourceConditionedDrafter(
    encoder_width=encoder_width,
    shared_width=shared_width,
    width=DRAFT_WIDTH,
    layers=DRAFT_LAYERS,
    heads=DRAFT_HEADS,
    ffn_width=DRAFT_FFN_WIDTH,
    max_positions=SOURCE_MAX_TOKENS + DRAFT_MAX_NEW_TOKENS + 2,
).to(DEVICE)
trainable_count = sum(parameter.numel() for parameter in drafter.parameters() if parameter.requires_grad)
print("Drafter trainable parameters:", f"{trainable_count:,}")
print("Drafter parameter dtype:", next(drafter.parameters()).dtype)

## Train the drafter with CE plus top-k distillation

For each teacher-generated sequence, the frozen target provides (1) token labels for cross-entropy and (2) its top-16 next-token logits for a truncated distillation KL. This top-k KL is an approximation to full-vocabulary KL. The optimizer sees drafter parameters only.

In [ ]:
optimizer = torch.optim.AdamW(drafter.parameters(), lr=LEARNING_RATE, weight_decay=0.01)


def as_training_tensors(trace):
    ids = torch.tensor(trace[: TEACHER_MAX_NEW_TOKENS + 1], dtype=torch.long, device=DEVICE).unsqueeze(0)
    if ids.shape[1] < 2:
        raise ValueError("Teacher trace must contain a decoder start token and at least one continuation token")
    decoder_ids = ids[:, :-1]
    labels = ids[:, 1:].clone()
    decoder_mask = decoder_ids.ne(PAD_ID)
    decoder_mask[:, 0] = True  # T5-family decoder start IDs can equal PAD.
    labels[labels.eq(PAD_ID)] = -100
    return decoder_ids, labels, decoder_mask


@torch.no_grad()
def encode_target_source(batch):
    with torch.autocast(device_type="cuda", dtype=DTYPE):
        output = encoder(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], return_dict=True)
    return output.last_hidden_state


train_order = list(range(len(train_sources)))
train_history = []
for epoch in range(TRAIN_EPOCHS):
    random.Random(SEED + epoch).shuffle(train_order)
    drafter.train()
    epoch_loss = 0.0
    started = time.time()
    progress = tqdm(train_order, desc=f"Drafter epoch {epoch + 1}/{TRAIN_EPOCHS}")
    for step, row_index in enumerate(progress, start=1):
        source_batch = tokenize_source(train_sources[row_index])
        source_memory = encode_target_source(source_batch)
        decoder_ids, labels, decoder_mask = as_training_tensors(train_traces[row_index])
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=DTYPE):
            teacher_output = target(
                encoder_outputs=BaseModelOutput(last_hidden_state=source_memory),
                attention_mask=source_batch["attention_mask"],
                decoder_input_ids=decoder_ids,
                decoder_attention_mask=decoder_mask,
                use_cache=False,
                return_dict=True,
            )
            teacher_logits = teacher_output.logits.float()
            teacher_top_values, teacher_top_ids = teacher_logits.topk(KD_TOP_K, dim=-1)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            draft_logits = drafter(
                decoder_ids,
                decoder_mask,
                source_memory,
                source_batch["attention_mask"],
                shared_embedding,
                shared_lm_head,
            ).float()
        valid = labels.ne(-100)
        ce_loss = F.cross_entropy(
            draft_logits.reshape(-1, draft_logits.shape[-1]), labels.reshape(-1), ignore_index=-100
        )
        temperature = KD_TEMPERATURE
        teacher_top_logp = F.log_softmax(teacher_top_values / temperature, dim=-1)
        teacher_top_prob = teacher_top_logp.exp()
        draft_logp_at_teacher_top = F.log_softmax(draft_logits / temperature, dim=-1).gather(-1, teacher_top_ids)
        token_kl = (teacher_top_prob * (teacher_top_logp - draft_logp_at_teacher_top)).sum(dim=-1)
        kd_loss = (token_kl * valid).sum() / valid.sum().clamp_min(1) * (temperature**2)
        loss = ce_loss + KD_WEIGHT * kd_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(drafter.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += float(loss.detach())
        progress.set_postfix(
            loss=f"{float(loss.detach()):.4f}", ce=f"{float(ce_loss.detach()):.4f}", kd=f"{float(kd_loss.detach()):.4f}"
        )
    mean_loss = epoch_loss / len(train_order)
    train_history.append({"epoch": epoch + 1, "mean_loss": mean_loss, "seconds": time.time() - started})
    print(f"Epoch {epoch + 1}: mean loss={mean_loss:.4f}, elapsed={time.time() - started:.1f}s")

# Confirm target tensors stayed frozen and optimizer only owns drafter parameters.
assert all(not parameter.requires_grad for parameter in target.parameters())
optimized_ids = {id(parameter) for group in optimizer.param_groups for parameter in group["params"]}
assert optimized_ids == {id(parameter) for parameter in drafter.parameters()}
print("Target remains frozen; optimizer updates drafter parameters only.")

## Exact greedy block verification (reference implementation)

The drafter proposes up to `DRAFT_K` tokens autoregressively. The frozen target verifies them in one full-prefix forward pass; at the first mismatch, the target's greedy token replaces the draft token. If every proposal is accepted, the verifier may emit one target bonus token.

This deliberately uses a simple full-prefix reference verifier without KV-cache rollback. It checks the acceptance logic and output parity, but it is **not** the optimized serving path and its timing must not be presented as a speedup over cached target decoding.

In [ ]:
@torch.inference_mode()
def target_encoder_memory(source_text):
    batch = tokenize_source(source_text)
    memory = encode_target_source(batch)
    return batch, memory


@torch.inference_mode()
def target_logits_for_prefix(batch, memory, decoder_ids):
    decoder_mask = torch.ones_like(decoder_ids, dtype=torch.long)
    with torch.autocast(device_type="cuda", dtype=DTYPE):
        result = target(
            encoder_outputs=BaseModelOutput(last_hidden_state=memory),
            attention_mask=batch["attention_mask"],
            decoder_input_ids=decoder_ids,
            decoder_attention_mask=decoder_mask,
            use_cache=False,
            return_dict=True,
        )
    return result.logits.float()


@torch.inference_mode()
def ordinary_greedy_reference(source_text, max_new_tokens=DRAFT_MAX_NEW_TOKENS):
    batch = tokenize_source(source_text)
    generated = target.generate(
        **batch,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        use_cache=True,
    )[0].tolist()
    if not generated or generated[0] != start_id:
        generated = [start_id] + generated
    return generated


@torch.inference_mode()
def speculative_greedy_reference(source_text, k=DRAFT_K, max_new_tokens=DRAFT_MAX_NEW_TOKENS):
    batch, memory = target_encoder_memory(source_text)
    decoder_ids = torch.tensor([[start_id]], dtype=torch.long, device=DEVICE)
    accepted = 0
    proposed = 0
    verify_calls = 0
    while decoder_ids.shape[1] - 1 < max_new_tokens:
        remaining = max_new_tokens - (decoder_ids.shape[1] - 1)
        proposal_ids = []
        for _ in range(min(k, remaining)):
            draft_mask = torch.ones_like(decoder_ids, dtype=torch.long)
            with torch.autocast(device_type="cuda", dtype=DTYPE):
                draft_logits = drafter(
                    decoder_ids,
                    draft_mask,
                    memory,
                    batch["attention_mask"],
                    shared_embedding,
                    shared_lm_head,
                ).float()
            proposal = int(draft_logits[0, -1].argmax())
            proposal_ids.append(proposal)
            decoder_ids = torch.cat([decoder_ids, torch.tensor([[proposal]], dtype=torch.long, device=DEVICE)], dim=1)
            if proposal in EOS_IDS:
                break
        proposed += len(proposal_ids)
        proposal_tensor = torch.tensor([proposal_ids], dtype=torch.long, device=DEVICE)
        prefix_length = decoder_ids.shape[1] - len(proposal_ids)
        verify_input = torch.cat([decoder_ids[:, :prefix_length], proposal_tensor], dim=1)
        target_logits = target_logits_for_prefix(batch, memory, verify_input)
        verify_calls += 1
        next_target_ids = (
            target_logits[0, prefix_length - 1 : prefix_length - 1 + len(proposal_ids)].argmax(dim=-1).tolist()
        )
        mismatch = None
        stopped_on_eos = False
        for offset, (proposal, target_id) in enumerate(zip(proposal_ids, next_target_ids)):
            if proposal == target_id:
                accepted += 1
                if target_id in EOS_IDS:
                    stopped_on_eos = True
                    break
            else:
                mismatch = (offset, target_id)
                break
        if mismatch is not None:
            offset, correction = mismatch
            keep_length = prefix_length + offset
            decoder_ids = decoder_ids[:, :keep_length]
            decoder_ids = torch.cat([decoder_ids, torch.tensor([[correction]], dtype=torch.long, device=DEVICE)], dim=1)
            if correction in EOS_IDS:
                break
            continue
        if stopped_on_eos:
            break
        # Every proposal matched. The last verifier position predicts the bonus token.
        if decoder_ids.shape[1] - 1 < max_new_tokens:
            bonus = int(target_logits[0, -1].argmax())
            decoder_ids = torch.cat([decoder_ids, torch.tensor([[bonus]], dtype=torch.long, device=DEVICE)], dim=1)
            if bonus in EOS_IDS:
                break
    return decoder_ids[0].tolist(), {
        "accepted_draft_tokens": accepted,
        "proposed_draft_tokens": proposed,
        "verify_calls": verify_calls,
    }


drafter.eval()
parity_rows = []
for index in range(min(PARITY_EVAL_EXAMPLES, len(validation_sources))):
    source = validation_sources[index]
    reference_ids = ordinary_greedy_reference(source)
    speculative_ids, stats = speculative_greedy_reference(source)
    equal = reference_ids == speculative_ids
    parity_rows.append(
        {
            "index": index,
            "exact_token_match": equal,
            "generated_tokens": len(reference_ids) - 1,
            "accepted_draft_tokens": stats["accepted_draft_tokens"],
            "proposed_draft_tokens": stats["proposed_draft_tokens"],
            "acceptance_rate": stats["accepted_draft_tokens"] / max(1, stats["proposed_draft_tokens"]),
            "target_verify_calls": stats["verify_calls"],
        }
    )
    print(json.dumps(parity_rows[-1]))

print("All token sequences equal:", all(row["exact_token_match"] for row in parity_rows))
print(
    "The block verifier recomputes full prefixes; this pilot does not measure production latency or cached-decoding speedup."
)

## Save the drafter artifact

The checkpoint stores the small drafter state, optimizer state, model/tokenizer identity, training settings, and parity diagnostics. It does not store or overwrite the frozen T5Gemma2 weights.

In [ ]:
artifact = {
    "format": "encdec_drafter_colab_pilot",
    "target_model_id": MODEL_ID,
    "dataset": f"{DATASET_ID}:{DATASET_CONFIG}",
    "train_examples": TRAIN_LIMIT,
    "validation_examples": VALIDATION_LIMIT,
    "source_max_tokens": SOURCE_MAX_TOKENS,
    "teacher_max_new_tokens": TEACHER_MAX_NEW_TOKENS,
    "draft": {
        "width": DRAFT_WIDTH,
        "layers": DRAFT_LAYERS,
        "heads": DRAFT_HEADS,
        "ffn_width": DRAFT_FFN_WIDTH,
        "block_k": DRAFT_K,
        "kd_top_k": KD_TOP_K,
        "kd_temperature": KD_TEMPERATURE,
        "kd_weight": KD_WEIGHT,
    },
    "train_history": train_history,
    "parity": parity_rows,
    "state_dict": drafter.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}
checkpoint_path = OUTPUT_DIR / "t5gemma2_1b1b_drafter_pilot.pt"
torch.save(artifact, checkpoint_path)
with (OUTPUT_DIR / "parity.json").open("w", encoding="utf-8") as handle:
    json.dump(parity_rows, handle, ensure_ascii=False, indent=2)
print("Saved drafter checkpoint:", checkpoint_path)
print("Saved parity report:", OUTPUT_DIR / "parity.json")
print("Remember to copy these files to Drive if they need to survive Colab runtime shutdown.")

## What this pilot can and cannot establish

A token-for-token match confirms the greedy accept/correct logic for the tested examples. Acceptance rate indicates whether the drafter resembles the target on these short teacher-generated continuations. Neither result establishes an end-to-end latency win. A production comparison must use a KV-cache-aware verifier, include encoder and drafter costs, measure batch-one latency and serving goodput, and compare against optimized cached T5Gemma2 generation under the same output policy.